# Merging the INE housing and tourism tables

## 1. Setup

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## 2. A small helper to read the INE files

The INE exports have a few lines of text on top, several header rows and some notes at the bottom, and those sizes change from file to file. Instead of counting lines by hand, this helper reads the whole file and keeps only the lines that look like data rows (the ones that start with a region code such as `11A: Área Metropolitana do Porto`).

In [2]:
# Reads an INE csv and returns only the data rows, as text columns (X1, X2, ...).
# We convert to numbers later, file by file.
read_ine_rows <- function(path) {
  all_lines <- read_lines(path, locale = locale(encoding = "latin1"))

  # A data row starts with a code like "PT: ", "11A: " or "1D1: ".
  # The population file has a year before the code ("2021;PT: ..."), so we allow that too.
  is_data_row <- str_detect(all_lines, "^(\\d{4})?;?[0-9A-Z]{1,4}: ")

  read_delim(
    I(all_lines[is_data_row]),
    delim = ";",
    col_names = FALSE,
    col_types = cols(.default = col_character()),
    show_col_types = FALSE
  )
}

# Turns text like "1460,8" into the number 1460.8 (the INE uses a decimal comma).
# parse_number leaves floating point noise (9.14 becomes 9.139999999999999), so we round
# to 2 decimals, the most the INE tables use.
to_number <- function(x) round(parse_number(x, locale = locale(decimal_mark = ",")), 2)

## 3. Region lookup

Every file labels regions as `"code: name"`. We read the "new homes" file first just to build one clean list of regions. We keep the **NUTS III** level (3-character codes like `11A` or `1D1`), plus the two island regions, whose code in most files is just `20` and `30`.

In [3]:
homes_raw <- read_ine_rows("data/viviendas_concluidas.csv")

regions <- homes_raw |>
  select(X1) |>
  separate(X1, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  filter(nchar(nuts_code) == 3 | nuts_code %in% c("20", "30"))

regions

nuts_code,region
<chr>,<chr>
111,Alto Minho
112,Cávado
119,Ave
11A,Área Metropolitana do Porto
11B,Alto Tâmega e Barroso
11C,Tâmega e Sousa
11D,Douro
11E,Terras de Trás-os-Montes
191,Região de Aveiro


## 4. Wide tables with one value per year

Four files have the same layout: one row per region and one column per year. For each one we name the year columns, drop the empty trailing column, melt to long format and keep only the regions we care about.

### Monthly wage

In [4]:
wage_raw <- read_ine_rows("data/salario_medio.csv")

wage_years <- 2024:2021   # the years, in the same order as the file columns

wage <- wage_raw |>
  select(1:(length(wage_years) + 1)) |>
  set_names(c("region_raw", wage_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "wage_monthly") |>
  mutate(year = as.integer(year), wage_monthly = to_number(wage_monthly)) |>
  select(nuts_code, year, wage_monthly)

wage

nuts_code,year,wage_monthly
<chr>,<int>,<dbl>
PT,2024,1576.0
PT,2023,1460.8
PT,2022,1362.4
PT,2021,1289.5
1,2024,1582.7
1,2023,1466.7
1,2022,1368.0
1,2021,1294.1
11,2024,1474.7


### Median rent per m²

In [5]:
rent_raw <- read_ine_rows("data/renta_mediana_m2.csv")

rent_years <- 2025:2020

rent <- rent_raw |>
  select(1:(length(rent_years) + 1)) |>
  set_names(c("region_raw", rent_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "rent_m2") |>
  mutate(year = as.integer(year), rent_m2 = to_number(rent_m2)) |>
  # This file lists the Azores twice (codes 20 and 200). We keep 20, like in the other files.
  filter(nuts_code != "200") |>
  select(nuts_code, year, rent_m2)

rent

nuts_code,year,rent_m2
<chr>,<int>,<dbl>
PT,2025,9.29
PT,2024,8.47
PT,2023,7.62
PT,2022,6.88
PT,2021,6.31
PT,2020,5.79
1,2025,9.32
1,2024,8.50
1,2023,7.64


### New homes completed

In [6]:
homes_years <- 2025:2011

homes <- homes_raw |>
  select(1:(length(homes_years) + 1)) |>
  set_names(c("region_raw", homes_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "homes_completed") |>
  mutate(year = as.integer(year), homes_completed = to_number(homes_completed)) |>
  select(nuts_code, year, homes_completed)

homes

nuts_code,year,homes_completed
<chr>,<int>,<dbl>
PT,2025,27301
PT,2024,24666
PT,2023,22280
PT,2022,21047
PT,2021,19926
PT,2020,16923
PT,2019,12954
PT,2018,10751
PT,2017,8213


### Foreign residents

In [7]:
foreign_raw <- read_ine_rows("data/extranjeros_residentes.csv")

foreign_years <- 2023:2021

foreign <- foreign_raw |>
  select(1:(length(foreign_years) + 1)) |>
  set_names(c("region_raw", foreign_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "foreign_residents") |>
  mutate(year = as.integer(year), foreign_residents = to_number(foreign_residents)) |>
  select(nuts_code, year, foreign_residents)

foreign

nuts_code,year,foreign_residents
<chr>,<int>,<dbl>
PT,2023,1044238
PT,2022,781247
PT,2021,698536
1,2023,1023941
1,2022,764349
1,2021,683669
11,2023,174958
11,2022,119385
11,2021,100084


## 5. Resident population (yearly estimates, by age)

The INE publishes a yearly estimate of the resident population on 31 December, split into five-year age groups. We use it instead of the Census 2021 count because it gives a population for every year, which is the right denominator for per-capita rates. The two do not match: the yearly estimate is built from administrative records and is higher than the Census in the regions that grew fastest (about 9% more in the Algarve already in 2021). The Census file stays in `data/poblacion_censo2021.csv` but is no longer used.

The year is written only on the first row of each block of regions, so we fill it downwards. We keep the total population and turn the age groups into shares of the population (in %).

In [8]:
population_raw <- read_ine_rows("data/poblacion_anual_edad.csv")

# Columns: X1 year (only on the first row of each block), X2 region, X3 total population
# (both sexes) and X4-X21 the 18 five-year age groups (0-4 ... 85 and over).
# The columns for men and for women that come after are not used.
population_age <- population_raw |>
  mutate(year = as.integer(X1)) |>
  fill(year) |>
  select(year, region_raw = X2, num_range("X", 3:21)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(num_range("X", 3:21), to_number))

population <- population_age |>
  transmute(
    nuts_code, year,
    population   = X3,
    share_0_14   = (X4 + X5 + X6) / X3 * 100,
    share_15_24  = (X7 + X8) / X3 * 100,
    share_25_34  = (X9 + X10) / X3 * 100,
    share_35_44  = (X11 + X12) / X3 * 100,
    share_45_54  = (X13 + X14) / X3 * 100,
    share_55_64  = (X15 + X16) / X3 * 100,
    share_65_79  = (X17 + X18 + X19) / X3 * 100,
    share_80plus = (X20 + X21) / X3 * 100,
    share_20_34  = (X8 + X9 + X10) / X3 * 100   # young adults, the age of most new tenants
  ) |>
  mutate(across(starts_with("share_"), ~ round(.x, 2)))

population

nuts_code,year,population,share_0_14,share_15_24,share_25_34,share_35_44,share_45_54,share_55_64,share_65_79,share_80plus,share_20_34
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PT,2025,11424031,12.36,10.30,12.80,13.33,14.75,13.13,16.30,7.04,18.24
1,2025,10912573,12.34,10.27,12.84,13.32,14.70,13.06,16.34,7.13,18.27
11,2025,3790554,11.79,10.16,12.19,12.83,14.86,14.16,17.21,6.81,17.56
111,2025,244343,11.41,9.57,10.90,12.25,14.35,13.53,18.83,9.16,15.90
112,2025,449945,12.81,10.92,12.65,13.64,15.47,13.66,15.18,5.68,18.37
119,2025,441329,12.18,10.06,12.15,13.00,15.19,14.64,16.88,5.89,17.46
11A,2025,1861727,11.79,10.11,12.72,13.19,14.98,13.93,16.86,6.41,18.08
11B,2025,83364,9.71,8.14,8.35,10.11,12.89,14.54,23.87,12.40,12.56
11C,2025,416728,12.11,10.91,12.28,12.62,15.12,15.20,16.12,5.64,17.99


## 5b. More yearly tables: net migration, median income and building permits

Net migration and the median declared income have the same layout as the tables in section 4 (one column per year). The median income comes from tax returns, per taxpayer and per year, and is only used as a check (section 9). Building permits look like the sale prices: 5 columns per year (Total, T0 or T1, T2, T3, T4 or more), and we keep only Total. All three keep every year the file has, so section 7b can use the long permit series.

In [9]:
migration_raw <- read_ine_rows("data/saldo_migratorio.csv")

migration_years <- 2025:2021

net_migration <- migration_raw |>
  select(1:(length(migration_years) + 1)) |>
  set_names(c("region_raw", migration_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "net_migration") |>
  mutate(year = as.integer(year), net_migration = to_number(net_migration)) |>
  select(nuts_code, year, net_migration)

income_raw <- read_ine_rows("data/ingreso_mediano.csv")

income_years <- 2024:2018

income <- income_raw |>
  select(1:(length(income_years) + 1)) |>
  set_names(c("region_raw", income_years)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "income_median") |>
  mutate(year = as.integer(year), income_median = to_number(income_median)) |>
  select(nuts_code, year, income_median)

net_migration
income

nuts_code,year,net_migration
<chr>,<int>,<dbl>
PT,2025,70862
PT,2024,216629
PT,2023,307288
PT,2022,371277
PT,2021,250079
1,2025,66855
1,2024,210034
1,2023,300560
1,2022,364980


nuts_code,year,income_median
<chr>,<int>,<dbl>
PT,2024,12666
PT,2023,11845
PT,2022,11103
PT,2021,10540
PT,2020,10023
PT,2019,9880
PT,2018,9431
1,2024,12659
1,2023,11842


In [10]:
permits_raw <- read_ine_rows("data/viviendas_licenciadas.csv")

permits_years <- 2025:2002
permits_sizes <- c("total", "t0t1", "t2", "t3", "t4plus")

permits_col_names <- paste0(rep(permits_years, each = length(permits_sizes)), "_", permits_sizes)

permits <- permits_raw |>
  select(1:(length(permits_col_names) + 1)) |>
  set_names(c("region_raw", permits_col_names)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year_size", values_to = "homes_licensed") |>
  separate(year_size, into = c("year", "size"), sep = "_") |>
  filter(size == "total") |>
  mutate(year = as.integer(year), homes_licensed = to_number(homes_licensed)) |>
  select(nuts_code, year, homes_licensed)
permits

nuts_code,year,homes_licensed
<chr>,<int>,<dbl>
PT,2025,42066
PT,2024,35371
PT,2023,33560
PT,2022,32232
PT,2021,30094
PT,2020,26168
PT,2019,25824
PT,2018,21896
PT,2017,14967


## 5c. Census 2021 tenure and the 2022 housing stock

These two tables have a single year, so they become fixed columns repeated on every row of the region:

- `households_2021` and `rented_households_2021`: households living in their usual home, in total and those who rent (Census 2021).
- `dwellings_2022`: all classic family dwellings in 2022 (the INE estimate of the housing stock).

In [11]:
tenure_raw <- read_ine_rows("data/hogares_regimen_2021.csv")

# Columns after the region: total, owned, rented, other arrangement.
tenure <- tenure_raw |>
  select(region_raw = X1, households_2021 = X2, rented_households_2021 = X4) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(c(households_2021, rented_households_2021), to_number)) |>
  select(nuts_code, households_2021, rented_households_2021)

stock_raw <- read_ine_rows("data/parque_viviendas_2022.csv")

stock <- stock_raw |>
  select(region_raw = X1, dwellings_2022 = X2) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(dwellings_2022 = to_number(dwellings_2022)) |>
  select(nuts_code, dwellings_2022)

tenure
stock

nuts_code,households_2021,rented_households_2021
<chr>,<dbl>,<dbl>
PT,4142581,922810
1,3962715,891434
11,1379430,306252
111,90671,13534
112,149713,27256
119,153389,34832
11A,681349,186037
11B,35735,3427
11C,148102,25918


nuts_code,dwellings_2022
<chr>,<dbl>
PT,6003809
1,5757767
11,1908900
111,156185
112,202322
119,200498
11A,842440
11B,73457
11C,205637


## 6. Median sale price per m²

This one has 5 columns for every year: Total, T0 or T1, T2, T3 and T4 or more. We build the column names by hand (year + size) so we can find each column, and then keep only **Total**.

Total is the INE's median sale price per m² over *all* the dwellings sold in the region. It is not the average of the four sizes (a median of all sales cannot be rebuilt from the medians of the groups), so it is the right single indicator and we drop the four size columns.

In [12]:
sales_raw <- read_ine_rows("data/precio_venta_m2.csv")

sales_years <- 2025:2019
sales_sizes <- c("total", "t0t1", "t2", "t3", "t4plus")

sales_col_names <- paste0(rep(sales_years, each = length(sales_sizes)), "_", sales_sizes)

sales <- sales_raw |>
  select(1:(length(sales_col_names) + 1)) |>
  set_names(c("region_raw", sales_col_names)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "year_size", values_to = "price") |>
  separate(year_size, into = c("year", "size"), sep = "_") |>
  filter(size == "total") |>
  mutate(year = as.integer(year), sale_price_m2 = to_number(price)) |>
  select(nuts_code, year, sale_price_m2)
sales

nuts_code,year,sale_price_m2
<chr>,<int>,<dbl>
PT,2025,2076
PT,2024,1777
PT,2023,1611
PT,2022,1484
PT,2021,1297
PT,2020,1190
PT,2019,1094
1,2025,2083
1,2024,1778


## 7. Overnight stays (monthly -\> yearly)

The tourism file uses the same **NUTS 2024** map as the other files and has one column per month ("Agosto de 2026", "Julho de 2026", ...). Two things to do here:

1.  Keep only the regions in our list (this drops the country, NUTS I / NUTS II rows and the duplicated Azores row `200`).
2.  Add up the 12 months of each year. We only keep 2022-2025: the file starts in January 2022 and 2026 is incomplete.

In [13]:
stays_lines <- read_lines(
  "data/turismo_noches.csv",
  locale = locale(encoding = "latin1")
)

# The header line with the months is the one that starts with ";Agosto de 2026;Julho de 2026..."
month_header <- stays_lines[str_detect(stays_lines, "^;[A-Za-zç]+ de \\d{4};")]
month_names  <- str_split(month_header, ";")[[1]]
month_names  <- month_names[month_names != ""]   # drop the empty bits

stays_raw <- read_ine_rows("data/turismo_noches.csv")

stays_monthly <- stays_raw |>
  select(1:(length(month_names) + 1)) |>
  set_names(c("region_raw", month_names)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "month", values_to = "tourist_nights") |>
  mutate(
    year = as.integer(str_extract(month, "\\d{4}")),
    tourist_nights = to_number(tourist_nights)
  )
stays_monthly

nuts_code,region,month,tourist_nights,year
<chr>,<chr>,<chr>,<dbl>,<int>
PT,Portugal,Agosto de 2026,10886445,2026
PT,Portugal,Julho de 2026,9583880,2026
PT,Portugal,Junho de 2026,8131586,2026
PT,Portugal,Maio de 2026,7996616,2026
PT,Portugal,Abril de 2026,7150714,2026
PT,Portugal,Março de 2026,5625115,2026
PT,Portugal,Fevereiro de 2026,4203862,2026
PT,Portugal,Janeiro de 2026,3741952,2026
PT,Portugal,Dezembro de 2025,4260786,2025


The codes in the tourism file already match the other files (including Lisbon, which is split into `1A0` and `1B0`), so no translation table is needed. We just join with the region list.

In [14]:
stays <- stays_monthly |>
  inner_join(regions |> select(nuts_code), by = "nuts_code") |>   # also drops country / NUTS I / NUTS II rows
  filter(year >= 2022, year <= 2025) |>                          # full years only
  group_by(nuts_code, year) |>
  summarise(tourist_nights = sum(tourist_nights), .groups = "drop")

stays

nuts_code,year,tourist_nights
<chr>,<int>,<dbl>
111,2022,860775
111,2023,928528
111,2024,985240
111,2025,1031079
112,2022,1058649
112,2023,1119300
112,2024,1167899
112,2025,1183221
119,2022,584132


## 7b. Historical series for the trend charts

The figure has two time-series charts that need more years than the main table: sale prices since 2019 and construction since 2002. We save them as two small, separate CSV files, with Portugal (`PT`) as a reference line:

- `data/precios_historicos.csv`: median sale price per m² (2019-2025) and median rent per m² (2020-2025, empty in 2019).
- `data/construccion_historica.csv`: homes licensed (2002-2025) and homes completed (2011-2025, empty before).

In [15]:
history_regions <- bind_rows(tibble(nuts_code = "PT", region = "Portugal"), regions)

price_history <- expand_grid(history_regions, year = 2019:2025) |>
  left_join(sales, by = c("nuts_code", "year")) |>
  left_join(rent,  by = c("nuts_code", "year")) |>
  arrange(nuts_code, year)

construction_history <- expand_grid(history_regions, year = 2002:2025) |>
  left_join(permits, by = c("nuts_code", "year")) |>
  left_join(homes,   by = c("nuts_code", "year")) |>
  arrange(nuts_code, year)

write_csv(price_history, "data/precios_historicos.csv")
write_csv(construction_history, "data/construccion_historica.csv")

# 27 codes (Portugal + 26 regions): 27 x 7 = 189 and 27 x 24 = 648 rows.
nrow(price_history)
nrow(construction_history)

[1] 189

[1] 648

## 8. Join everything

We start from a "skeleton" with every region for every year from 2021 to 2024, and then attach each table to it. Where a table has no data for a given region and year, the value stays `NA` (the equivalent of `NaN` in pandas).

Why 2021-2024? Each source covers different years: wage 2021-2024, median income 2018-2024, rent 2020-2025, sale price 2019-2025, new homes 2011-2025, building permits 2002-2025, foreign residents 2021-2023, tourism 2022-2025, population and net migration 2021-2025. No year has everything, but 2021-2024 has almost all of them in the four years. It also keeps 2021 as the starting point to see how rent affordability changed. The only gaps are tourism in 2021 and foreign residents in 2024.

The Census 2021 tenure and the 2022 housing stock have a single value per region, so they are joined by region only and repeated on every year.

In [16]:
wage

nuts_code,year,wage_monthly
<chr>,<int>,<dbl>
PT,2024,1576.0
PT,2023,1460.8
PT,2022,1362.4
PT,2021,1289.5
1,2024,1582.7
1,2023,1466.7
1,2022,1368.0
1,2021,1294.1
11,2024,1474.7


In [17]:
skeleton <- expand_grid(regions, year = 2021:2024)

merged <- skeleton |>
  left_join(wage,          by = c("nuts_code", "year")) |>
  left_join(income,        by = c("nuts_code", "year")) |>
  left_join(rent,          by = c("nuts_code", "year")) |>
  left_join(sales,         by = c("nuts_code", "year")) |>
  left_join(homes,         by = c("nuts_code", "year")) |>
  left_join(permits,       by = c("nuts_code", "year")) |>
  left_join(foreign,       by = c("nuts_code", "year")) |>
  left_join(population,    by = c("nuts_code", "year")) |>
  left_join(net_migration, by = c("nuts_code", "year")) |>
  left_join(stays,         by = c("nuts_code", "year")) |>
  left_join(tenure,        by = "nuts_code") |>   # Census 2021, repeated on every year
  left_join(stock,         by = "nuts_code") |>   # 2022, repeated on every year
  arrange(nuts_code, year)

glimpse(merged)

Rows: 104
Columns: 25
$ nuts_code              <chr> "111", "111", "111", "111", "112", "112", "112"…
$ region                 <chr> "Alto Minho", "Alto Minho", "Alto Minho", "Alto…
$ year                   <int> 2021, 2022, 2023, 2024, 2021, 2022, 2023, 2024,…
$ wage_monthly           <dbl> 1095.0, 1154.3, 1237.7, 1340.1, 1134.3, 1183.1,…
$ income_median          <dbl> 9427, 9993, 10755, 11646, 9897, 10474, 11240, 1…
$ rent_m2                <dbl> 4.29, 4.69, 5.35, 6.00, 5.14, 5.77, 6.59, 7.38,…
$ sale_price_m2          <dbl> 945, 1064, 1108, 1248, 1091, 1230, 1385, 1527, …
$ homes_completed        <dbl> 588, 596, 581, 656, 1362, 1706, 1465, 1471, 121…
$ homes_licensed         <dbl> 745, 888, 1108, 941, 1754, 2008, 2066, 2146, 17…
$ foreign_residents      <dbl> 7505, 9187, 13145, NA, 16781, 19947, 28650, NA,…
$ population             <dbl> 230770, 236830, 241424, 243852, 417412, 430802,…
$ share_0_14             <dbl> 11.57, 11.65, 11.58, 11.52, 13.54, 13.54, 13.29…
$ share_15_24     

In [18]:
merged

nuts_code,region,year,wage_monthly,income_median,rent_m2,sale_price_m2,homes_completed,homes_licensed,foreign_residents,⋯,share_45_54,share_55_64,share_65_79,share_80plus,share_20_34,net_migration,tourist_nights,households_2021,rented_households_2021,dwellings_2022
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
111,Alto Minho,2021,1095.0,9427,4.29,945,588,745,7505,⋯,14.34,14.05,18.75,8.96,15.03,124,NA,90671,13534,156185
111,Alto Minho,2022,1154.3,9993,4.69,1064,596,888,9187,⋯,14.28,13.80,18.63,8.78,15.60,7788,860775,90671,13534,156185
111,Alto Minho,2023,1237.7,10755,5.35,1108,581,1108,13145,⋯,14.28,13.61,18.65,8.78,16.00,6096,928528,90671,13534,156185
111,Alto Minho,2024,1340.1,11646,6.00,1248,656,941,NA,⋯,14.29,13.55,18.73,8.93,16.00,4102,985240,90671,13534,156185
112,Cávado,2021,1134.3,9897,5.14,1091,1362,1754,16781,⋯,15.62,13.66,14.31,5.38,17.78,-314,NA,149713,27256,202322
112,Cávado,2022,1183.1,10474,5.77,1230,1706,2008,19947,⋯,15.55,13.50,14.43,5.32,18.22,13745,1058649,149713,27256,202322
112,Cávado,2023,1276.5,11240,6.59,1385,1465,2066,28650,⋯,15.54,13.45,14.62,5.38,18.49,10181,1119300,149713,27256,202322
112,Cávado,2024,1403.3,12116,7.38,1527,1471,2146,NA,⋯,15.48,13.53,14.90,5.48,18.46,7051,1167899,149713,27256,202322
119,Ave,2021,1080.8,9506,3.76,976,1216,1799,6803,⋯,15.67,14.67,15.60,5.49,17.31,-296,NA,153389,34832,200498


## 9. Quick checks

How many rows do we have, and how many missing values per column? We expect 104 rows (26 regions x 4 years) and missing values only in `tourist_nights` (2021) and `foreign_residents` (2024): 26 each. Every other column, including the new ones, should be complete.

In [19]:
nrow(merged)
n_distinct(merged$nuts_code)

# There must be exactly one row per region and year (this should print 0).
merged |> count(nuts_code, year) |> filter(n > 1) |> nrow()

merged |>
  summarise(across(everything(), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "missing")

[1] 104

[1] 26

[1] 0

column,missing
<chr>,<int>
nuts_code,0
region,0
year,0
wage_monthly,0
income_median,0
rent_m2,0
sale_price_m2,0
homes_completed,0
homes_licensed,0


Do all the regions of each table exist in our region list? Each of these should show zero rows (otherwise a code did not match).

In [20]:
wage          |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
income        |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
rent          |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
sales         |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
permits       |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
foreign       |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
population    |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
net_migration |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
tenure        |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
stock         |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)
stays         |> anti_join(regions, by = "nuts_code") |> distinct(nuts_code)

# Every region should have 4 years of tourism data (2022-2025), Lisbon (1A0, 1B0) included.
stays |> count(nuts_code) |> filter(n != 4)

nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


...

nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>
PT
1
11
19
1D
1A
1B
1C
15


nuts_code
<chr>


nuts_code,n
<chr>,<int>


Spot checks against the original files:

- Porto Metropolitan Area (`11A`) in 2024: monthly wage 1620.1, rent 9.14, sale price 1986 and 5804 new homes.
- Grande Lisboa (`1A0`) in 2024: population 2,419,228, net migration 56,494, 1,072,111 dwellings in 2022 and 31.4% of households renting.
- Algarve (`150`) in 2024: population 577,095.

In [21]:
merged |>
  filter(nuts_code == "11A", year == 2024) |>
  select(nuts_code, year, wage_monthly, rent_m2, sale_price_m2, homes_completed)

merged |>
  filter(nuts_code %in% c("1A0", "150"), year == 2024) |>
  transmute(nuts_code, population, net_migration, dwellings_2022,
            rented_share = round(rented_households_2021 / households_2021 * 100, 1))

nuts_code,year,wage_monthly,rent_m2,sale_price_m2,homes_completed
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
11A,2024,1620.1,9.14,1986,5804


nuts_code,population,net_migration,dwellings_2022,rented_share
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
150,577095,14459,393387,23.0
1A0,2419228,56494,1072111,31.4


### Check: does the ranking change with the median income instead of the average wage?

The rent effort in the figure is the monthly rent of an 80 m² home divided by the average monthly wage. As a check we repeat it with the median declared income (annual, per taxpayer, from tax returns): rent x 12 / income. Its levels are higher, because the median is lower than the average and it includes pensioners, so we only compare the **ranking** of the regions with a Spearman correlation. A value close to 1 means the same regions come out on top either way.

In [22]:
effort_check <- merged |>
  filter(year %in% c(2021, 2024)) |>
  transmute(
    nuts_code, year,
    effort_wage   = rent_m2 * 80 / wage_monthly * 100,
    effort_income = rent_m2 * 80 * 12 / income_median * 100
  ) |>
  pivot_wider(names_from = year, values_from = c(effort_wage, effort_income))

# Same order of regions in 2024?
cor(effort_check$effort_wage_2024, effort_check$effort_income_2024, method = "spearman")

# Same regions getting worse between 2021 and 2024?
cor(effort_check$effort_wage_2024 - effort_check$effort_wage_2021,
    effort_check$effort_income_2024 - effort_check$effort_income_2021, method = "spearman")

[1] 0.9835897

[1] 0.9760684

## 10. Save the result

In [23]:
write_csv(merged, "data/datos_unidos.csv")